# PGMM infrastructure smoke test

Cheapest possible de-risking before any quota is committed. CPU-only, so it
does not touch the 30h/week GPU quota. Checks four assumptions the whole
plan rests on, each of which is invisible from outside Kaggle:

1. a kernel pushed through the API actually runs
2. internet is enabled (needs a phone-verified account) — without it the
   repo cannot be cloned
3. the LSA64 dataset attaches and holds the 3200 clips we expect
4. the repo installs and its 49 tests pass on Kaggle's image, not just locally

GPU visibility is checked separately, in a GPU-enabled run.

In [ ]:
# 1. environment
import subprocess, sys, platform
print("python  ", sys.version.split()[0])
print("platform", platform.platform())
import torch
print("torch   ", torch.__version__, "| cuda build:", torch.version.cuda)
print("gpus    ", torch.cuda.device_count(), "(expected 0 here: CPU-only run)")

In [ ]:
# 2. internet + clone. If internet is disabled this cell fails, and that is
#    the single most important thing this notebook can tell us.
REPO = "https://github.com/SonLamHG/pgmm.git"
BRANCH = "feat/m0-m1-harness"  # pinned: the repo's default branch will move

r = subprocess.run(
    ["git", "clone", "-q", "--branch", BRANCH, "--depth", "1", REPO, "/kaggle/working/repo"],
    capture_output=True, text=True,
)
print("clone exit:", r.returncode)
print(r.stdout or "", r.stderr or "")
assert r.returncode == 0, "clone failed -- internet disabled, or repo not public"
print("INTERNET OK, REPO CLONED")

In [ ]:
# 3. LSA64 attached? Re-verify D13 against the files themselves, not the API
#    listing -- this is the first time anything actually opens the archive.
#
#    Discover the mount point instead of hardcoding it. Run 1 of this notebook
#    died here on a guessed path (/kaggle/input/lsa64-dataset), and a guess
#    costs a whole push/queue/run cycle to disprove.
from pathlib import Path
import re

INPUT = Path("/kaggle/input")
print("attached datasets:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else "NONE")

mp4s = sorted(INPUT.rglob("*.mp4"))
assert mp4s, f"no .mp4 anywhere under {INPUT} -- is the dataset attached at all?"
print("mp4 count :", len(mp4s), "(expected 3200)")
print("sample    :", mp4s[0].relative_to(INPUT))
print("parent dir:", mp4s[0].parent)

rx = re.compile(r"^(\d{3})_(\d{3})_(\d{3})$")
parsed = [rx.match(p.stem) for p in mp4s]
bad = [p.stem for p, m in zip(mp4s, parsed) if not m]
assert not bad, f"unparseable names: {bad[:5]}"
signs = {int(m.group(1)) for m in parsed}
signers = {int(m.group(2)) for m in parsed}
reps = {int(m.group(3)) for m in parsed}
print(f"signs {min(signs)}..{max(signs)} ({len(signs)}) | "
      f"signers {min(signers)}..{max(signers)} ({len(signers)}) | "
      f"reps {min(reps)}..{max(reps)} ({len(reps)})")
assert len(mp4s) == 3200 and len(signs) == 64 and len(signers) == 10 and len(reps) == 5
print("LSA64 OK -- D13 confirmed against the real files")

In [ ]:
# 4. does our code install and pass its tests on Kaggle's image?
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print("pip exit:", r.returncode)
print((r.stderr or "")[-2000:])

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(r.stdout[-3000:])
print("pytest exit:", r.returncode, "(0 = all 49 pass on Kaggle too)")

In [ ]:
# 5. can one video actually be decoded here? cv2 on Kaggle's image is not the
#    one we tested against locally, and the whole pipeline hinges on decoding.
import sys
sys.path.insert(0, "/kaggle/working/repo")
from pgmm.data.lsa64_prepare import prepare_video

n = prepare_video(mp4s[0], Path("/kaggle/working/_probe"), size=128)
print("frames decoded from", mp4s[0].name, "->", n)
assert n > 0

import cv2
img = cv2.imread("/kaggle/working/_probe/frame_00000.jpg")
print("frame shape:", img.shape, "(expected (128, 128, 3))")
assert img.shape == (128, 128, 3)
print("DECODE OK")

In [ ]:
# 6. rough decode cost -> how long will preprocessing all 3200 clips take?
import time, shutil

shutil.rmtree("/kaggle/working/_probe", ignore_errors=True)
t0 = time.time()
total = sum(prepare_video(p, Path(f"/kaggle/working/_probe/{p.stem}"), size=128)
            for p in mp4s[:20])
dt = time.time() - t0
print(f"20 clips, {total} frames in {dt:.1f}s -> {dt/20:.2f}s/clip")
print(f"projected for 3200 clips: {dt/20*3200/60:.1f} min single-process")
print(f"mean frames/clip: {total/20:.1f} -> projected total frames ~{total/20*3200:,.0f}")
shutil.rmtree("/kaggle/working/_probe", ignore_errors=True)